In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

In [2]:
import pandas as pd
import numpy as np

def preprocessing(data, typ):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13",
                     
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    
    # Original aggregated features
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    # === NEW: Statistical Features for each category ===
    for prefix, count in [('M', 18), ('E', 20), ('I', 9), ('V', 13), ('S', 12), ('D', 9)]:
        cols = [f'{prefix}{i}' for i in range(1, count+1) if f'{prefix}{i}' in data.columns]
        if cols:
            data[f'{prefix}_std'] = data[cols].std(axis=1)
            data[f'{prefix}_mean'] = data[cols].mean(axis=1)
            data[f'{prefix}_max'] = data[cols].max(axis=1)
            data[f'{prefix}_min'] = data[cols].min(axis=1)
            data[f'{prefix}_range'] = data[f'{prefix}_max'] - data[f'{prefix}_min']
            data[f'{prefix}_cv'] = data[f'{prefix}_std'] / (abs(data[f'{prefix}_mean']) + 1e-8)
            
            main_features.extend([f'{prefix}_std', f'{prefix}_mean', f'{prefix}_max', 
                                f'{prefix}_min', f'{prefix}_range', f'{prefix}_cv'])

    # === NEW: Momentum and Trend Features ===
    # Assuming sequential ordering in features represents time progression
    for prefix, count in [('M', 18), ('E', 20), ('V', 13), ('S', 12)]:
        cols = [f'{prefix}{i}' for i in range(1, count+1) if f'{prefix}{i}' in data.columns]
        if len(cols) >= 3:
            # Simple momentum (last - first)
            data[f'{prefix}_momentum'] = data[cols[-1]] - data[cols[0]]
            # Acceleration (rate of change of change)
            mid = len(cols) // 2
            data[f'{prefix}_accel'] = (data[cols[-1]] - data[cols[mid]]) - (data[cols[mid]] - data[cols[0]])
            
            main_features.extend([f'{prefix}_momentum', f'{prefix}_accel'])

    # Original pairwise features
    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
    main_features.extend([f'ME_prod_M*_E*', f'ME_ratio_M*_E*'])
            
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_ratio_M*_I*'] = data['M*'] / (data['I*'] + 1e-8)
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    main_features.extend([f'MI_prod_M*_I*', f'MI_ratio_M*_I*', f'MI_spread_M*_I*'])
        
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    main_features.extend([f'MP_prod_M*_P*', f'MP_ratio_M*_P*'])
    
    data[f'MV_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
    if 'return' in 'M*'.lower():
        data[f'MV_sharpe_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
        main_features.append(f'MV_sharpe_M*_V*')
    main_features.extend([f'MV_ratio_M*_V*', f'MV_prod_M*_V*'])
        
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.extend([f'MS_prod_M*_S*', f'MS_weighted_M*_S*'])
        
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_ratio_E*_I*'] = data['E*'] / (data['I*'] + 1e-8)
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.extend([f'EI_diff_E*_I*', f'EI_ratio_E*_I*', f'EI_prod_E*_I*'])
        
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    data[f'EP_ratio_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.extend([f'EP_prod_E*_P*', f'EP_ratio_E*_P*'])
        
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.extend([f'EV_prod_E*_V*', f'EV_uncertainty_E*_V*'])
        
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.extend([f'ES_prod_E*_S*', f'ES_diff_E*_S*'])
        
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    data[f'IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.extend([f'IP_prod_I*_P*', f'IP_discount_I*_P*'])
    
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
        
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
        
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    data[f'PV_stability_P*_V*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.extend([f'PV_prod_P*_V*', f'PV_stability_P*_V*'])
        
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.extend([f'PS_prod_P*_S*', f'PS_contrarian_P*_S*'])
        
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.extend([f'VS_prod_V*_S*', f'VS_panic_V*_S*'])

    # Original composite features
    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 
    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 
    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 
    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']
    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    # === NEW: Advanced Composite Features ===
    
    # Risk-adjusted returns
    data['risk_adjusted_return'] = (data['M*'] - data['I*']) / (data['V*'] + 1e-8)
    main_features.append('risk_adjusted_return')
    
    # Market regime indicator
    data['regime_indicator'] = np.sign(data['M*']) * np.log1p(abs(data['V*']))
    main_features.append('regime_indicator')
    
    # Economic stress index
    data['economic_stress'] = abs(data['E*']) * data['V*'] * (1 - np.tanh(data['S*']))
    main_features.append('economic_stress')
    
    # Liquidity proxy
    data['liquidity_proxy'] = data['M*'] / (data['V*'] * abs(data['I*']) + 1e-8)
    main_features.append('liquidity_proxy')
    
    # Sentiment divergence (when sentiment differs from fundamentals)
    data['sentiment_divergence'] = abs(data['S*'] - np.sign(data['E*']) * abs(data['E*']))
    main_features.append('sentiment_divergence')
    
    # Price momentum quality
    data['momentum_quality'] = data['M*'] / (data['V*'] + abs(data['D*']) + 1e-8)
    main_features.append('momentum_quality')
    
    # Multi-factor risk score
    data['risk_score'] = np.sqrt(data['V*']**2 + data['D*']**2 + (data['I*']*0.5)**2)
    main_features.append('risk_score')
    
    # === NEW: Non-linear transformations ===
    for feat in ['M*', 'E*', 'V*', 'S*']:
        data[f'{feat}_squared'] = data[feat] ** 2
        data[f'{feat}_log'] = np.sign(data[feat]) * np.log1p(abs(data[feat]))
        data[f'{feat}_sqrt'] = np.sign(data[feat]) * np.sqrt(abs(data[feat]))
        
        main_features.extend([f'{feat}_squared', f'{feat}_log', f'{feat}_sqrt'])
    
    # === NEW: Interaction complexity features ===
    # Three-way interactions
    data['MEIV_quad'] = data['M*'] * data['E*'] * data['I*'] / (data['V*'] + 1e-8)
    data['MPVS_quad'] = data['M*'] * data['P*'] * data['V*'] * data['S*']
    data['EIVD_quad'] = data['E*'] * data['I*'] / ((data['V*'] * data['D*']) + 1e-8)
    
    main_features.extend(['MEIV_quad', 'MPVS_quad', 'EIVD_quad'])
    
    # === NEW: Rolling statistics proxies (using cross-sectional variance) ===
    # Dispersion measures
    all_m_cols = [c for c in data.columns if c.startswith('M') and c[1:].isdigit()]
    all_e_cols = [c for c in data.columns if c.startswith('E') and c[1:].isdigit()]
    
    if all_m_cols:
        data['M_dispersion'] = data[all_m_cols].var(axis=1)
        main_features.append('M_dispersion')
    
    if all_e_cols:
        data['E_dispersion'] = data[all_e_cols].var(axis=1)
        main_features.append('E_dispersion')
    
    # === NEW: Relative strength features ===
    data['M_dominance'] = data['M*'] / (data['M*'] + data['E*'] + data['I*'] + 1e-8)
    data['E_dominance'] = data['E*'] / (data['M*'] + data['E*'] + data['I*'] + 1e-8)
    data['V_dominance'] = data['V*'] / (data['V*'] + data['S*'] + data['D*'] + 1e-8)
    
    main_features.extend(['M_dominance', 'E_dominance', 'V_dominance'])
    
    # === NEW: Polynomial features for key ratios ===
    key_ratios = ['ME_ratio_M*_E*', 'MV_ratio_M*_V*', 'EI_ratio_E*_I*']
    for ratio in key_ratios:
        if ratio in data.columns:
            data[f'{ratio}_sq'] = data[ratio] ** 2
            data[f'{ratio}_cb'] = data[ratio] ** 3
            main_features.extend([f'{ratio}_sq', f'{ratio}_cb'])
    
    # Filter to relevant columns
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else: 
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()

    for col in data.columns:
        if col != 'target':
            data[col].fillna(data[col].mean(), inplace=True)
    
    return data


train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
train_processed = preprocessing(train, "train")
train_x = train_processed.drop(columns=["target"])
train_y = train_processed['target']

In [3]:
train_processed

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,M_dominance,E_dominance,V_dominance,ME_ratio_M*_E*_sq,ME_ratio_M*_E*_cb,MV_ratio_M*_V*_sq,MV_ratio_M*_V*_cb,EI_ratio_E*_I*_sq,EI_ratio_E*_I*_cb,target
6969,0.682135,0.017196,0.007937,0.007937,0.007937,0.007937,0.048280,1.148054,1.303514,1.046752,...,-0.026094,0.506934,0.869541,0.002650,-0.000136,0.006450,-0.000518,0.953456,0.931003,0.001145
6970,0.681101,0.016865,0.007606,0.007606,0.007606,0.007606,0.008267,1.146588,1.303443,1.047816,...,0.013001,0.494214,0.595480,0.000692,0.000018,0.001892,0.000082,1.005810,1.008728,0.004738
6971,0.680068,0.016534,0.007275,0.007275,0.007275,0.007275,0.007937,1.145124,1.303371,1.048881,...,-0.064092,0.530037,0.500371,0.014622,-0.001768,0.037856,-0.007366,0.985010,0.977599,0.006016
6972,0.679035,0.016204,0.006944,0.006944,0.006944,0.006944,0.007606,1.120467,2.311946,1.049948,...,-0.187362,0.644882,0.410023,0.084412,-0.024525,0.355362,-0.211839,1.413162,1.679917,0.001414
6973,0.678003,0.015873,0.006614,0.006614,0.006614,0.006614,0.007275,1.119052,2.308384,1.051017,...,-0.123619,0.608619,0.468882,0.041256,-0.008380,0.166704,-0.068064,1.396608,1.650486,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,1.565379,0.184524,0.019180,0.019180,0.005952,0.005952,0.911376,-0.083496,-0.572447,0.223638,...,0.072714,0.591611,0.547162,0.015107,0.001857,0.072515,0.019527,3.106255,5.474640,0.002457
8986,1.562946,0.184193,0.018849,0.018849,0.005622,0.005622,0.911706,-0.083542,-0.572080,0.222910,...,0.162135,0.523528,0.391251,0.095912,0.029704,0.447655,0.299513,2.773891,4.619916,0.002312
8987,1.560520,0.183862,0.018519,0.018519,0.005291,0.005291,0.912037,-0.083874,-0.572016,0.222211,...,0.139036,0.539376,0.342982,0.066447,0.017128,0.425508,0.277563,2.813101,4.718219,0.002891
8988,1.558102,0.183532,0.018188,0.018188,0.004960,0.004960,0.912368,-0.084206,-0.571952,0.221513,...,0.078874,0.580985,0.618617,0.018430,0.002502,0.113211,0.038092,2.917510,4.983318,0.008310


In [4]:
import numpy as np
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

class ResidualBoostingEnsemble:
    
    def __init__(self, base_params, n_models=3):
        self.base_params = base_params
        self.n_models = n_models
        self.models = []
        
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        current_train_pred = np.zeros(len(train_y))
        
        for i in range(self.n_models):
            print(f'Training Model {i+1}...')
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            model = LGBMRegressor(**self.base_params)
            model.fit(train_x, target)
            self.models.append(model)
            
            pred = model.predict(train_x)
            current_train_pred += pred
            
            train_mape = mean_absolute_error(train_y, current_train_pred)
            print(f'Cumulative Training MAPE: {train_mape:.4f}')
        
        return self
    
    def predict(self, X):
        predictions = np.zeros(len(X))
        for model in self.models:
            predictions += model.predict(X)
        return predictions
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f'Model saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f'MAPE: {mape:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'RMSE: {rmse:.4f}')
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse}


LGBM_R_parm = {
               'boosting_type': 'gbdt', 
               'colsample_bytree': 0.95, 
               'learning_rate': 0.08,
               'max_bin': 100,
               'max_depth': 12,
               'metric': 'regression_l2', 
               'min_child_samples': 60,
               'min_data_in_leaf': 20, 
               'n_estimators': 7000,
               'num_leaves': 50,
               'objective': 'mean_absolute_error', 
               'reg_alpha': 0.8,
               'reg_lambda': 3.5, 
               'subsample': 0.75, 
               'verbosity': -1
              }

ensemble = ResidualBoostingEnsemble(base_params=LGBM_R_parm, n_models=3)
ensemble.fit(train_x, train_y)

ensemble.save('LGBM_Residual_Ensemble.pkl')

print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

LGBM_R_model = joblib.load('LGBM_Residual_Ensemble.pkl')

Training Model 1...
Cumulative Training MAPE: 0.0018
Training Model 2...
Cumulative Training MAPE: 0.0016
Training Model 3...
Cumulative Training MAPE: 0.0015
Model saved to LGBM_Residual_Ensemble.pkl

Final Training Evaluation:
MAPE: 82046928.2425
MAE: 0.0015
RMSE: 0.0040


In [5]:
SIGNAL_MULTIPLIER = 5000.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1 , MIN_SIGNAL, MAX_SIGNAL)

def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        
        test_processed = preprocessing(test_df, 'inference')
        LGBM_R_y_pred = np.float64(LGBM_R_model.predict(test_processed)[0])
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        print(np.float64(signal))
        return np.float64(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

0.0
0.0
2.0
2.0
0.0
2.0
2.0
2.0
2.0
1.4941906161523721
